# CRTC - Google Colab Setup

Play a CR training camp on colab with numeric states and video recording!

1. Click `Run all` to to see it in action
2. Modify [6.](#6-run-a-game-log-states-and-collect-frames) to try out different decks and control the battle


## 1. Clone the source


In [ ]:
!git clone https://github.com/b10902118/crtc.git
%cd crtc

## 2. Download the CR APK

The link is slow. Better to use your own copy


In [ ]:
!DOWNLOAD_URL=$(wget -qO- https://tinyurl.com/freeapk-mobi-cr-1-9-2-apk | grep download_button | grep -oP 'href="\K[^"]+') && \
wget -O "1.9.2.apk" "$DOWNLOAD_URL"

## 3. Install build and runtime dependencies for 32-bit (i386)


In [ ]:
!sudo dpkg --add-architecture i386
!sudo apt-get update
!sudo apt-get install -y \
    build-essential \
    gcc-multilib \
    g++-multilib \
    libegl1-mesa-dev:i386 \
    libgles2-mesa-dev:i386 \
    cmake \
    clang \
    python3 \
    patchelf \
    unzip

## 4. Run the installation script


In [ ]:
!./scripts/install.sh --graphics

## 5. Install `crtc` python package


In [ ]:
%pip install -q .

## 6. Run a game. Log states and collect frames

Colab's cpu rendering is slow. Remove `render_mode` and it will be much faster


In [ ]:
from crtc import CRTC, TRAINER, PLAYER
from crtc.utils import print_observation
import time

from PIL import Image
from IPython.display import display

env = CRTC(render_mode="rgb_array")
observations, infos = env.reset(
    options={
        "decks": {
            TRAINER: [
                {"name": "HogRider", "level": 7},
                {"name": "IceGolemite"},  # default standard level
                {"name": "Cannon"},
                {"name": "IceSpirits"},
                {"name": "Skeletons"},
                {"name": "Log"},
                {"name": "Fireball"},
                {"name": "Musketeer"},
            ],
            PLAYER: [
                {"name": "Xbow"},
                {"name": "Knight"},
                {"name": "Archer"},
                {"name": "IceSpirits"},
                {"name": "Tesla"},
                {"name": "Fireball"},
                {"name": "Skeletons"},
                {"name": "Log"},
            ],
        }
    }
)
print_observation(observations)

# play the first card at each player's left bridge
env.step(((0, 0, 16), (0, 0, 16)))

imgs = []
for t in range(2, 600):  # simulate 600 ticks
    observations, _, terms, _, infos = env.step(((-1, 0, 0), (-1, 0, 0)))
    if t % 10 == 0:  # 2 fps
        print_observation(observations)
        imgs.append(env.render())
env.close()

## 7. Get the game video


In [ ]:
import imageio

writer = imageio.get_writer(
    "output.mp4",
    fps=10,
    codec="libx264",
    quality=None,  # Disable default quality presets to use custom ffmpeg parameters
    ffmpeg_params=[
        "-crf",
        "23",
        "-pix_fmt",
        "yuv420p",
    ],  # yuv420p ensures it plays everywhere
)
for img in imgs:
    writer.append_data(img)
writer.close()

In [ ]:
from IPython.display import Video

Video("output.mp4", embed=True)